In [1]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
pip install datasets pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
#Load dataset from HF Hub: AlekseyKorshuk/quora-question-pairs
from datasets import load_dataset

dataset = load_dataset("AlekseyKorshuk/quora-question-pairs")

df = dataset["train"].to_pandas()

In [4]:
df.shape

(404290, 6)

In [5]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [6]:
new_df = df.sample(30000)

In [7]:
new_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

In [8]:
ques_df = new_df[['question1', 'question2']]
ques_df.head()

,question1,question2
264329,How do I reduce my anger?,How can I control my anger?
114087,How can I recover my Gmail forgot my password ...,How can I reset my Gmail password if I don't r...
361623,What's that one change you made in your routin...,What is the one risk you should have taken tha...
292385,"Are YouTube, Vevo, Apple Music and Spotify the...",What is music streaming cms I can use to build...
140308,Why do I feel sleepy and tired after consuming...,Why do I feel dizzy after a flu?


In [9]:
# Check how many None/NaN
print(new_df[['question1','question2']].isnull().sum())

# Drop or fill
new_df = new_df.dropna(subset=['question1','question2']) # Option 1: Remove
# OR
new_df = new_df.fillna('') # Option 2: Fill with blank

question1    0
question2    0
dtype: int64


In [10]:
from sklearn.feature_extraction.text import CountVectorizer
# Extracting question text columns
ques_df = new_df[['question1', 'question2']]

# Merging question lists to learn vocabulary from both sets
questions = list(ques_df['question1']) + list(ques_df['question2'])

# Creating BoW features - limiting to 3000 features to reduce dimensionality
cv = CountVectorizer(max_features = 3000)
# Transforming text to vectors and splitting back into q1 and q2 feature matrices
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questions).toarray(),2)

In [11]:
#Convert to Dataframe
temp_df1 = pd.DataFrame(q1_arr,index = ques_df.index)
temp_df2 = pd.DataFrame(q2_arr,index = ques_df.index)
# concatinate sie by side
temp_df = pd.concat([temp_df1,temp_df2], axis=1)
temp_df.shape

(30000, 6000)

In [12]:
temp_df

,0,1,2,3,4,5,6,7,8,9,...,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999
264329,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
114087,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
361623,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
292385,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
140308,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48202,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86912,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
142117,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
307630,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
# add new colum "is_duplicate"
temp_df['is_duplocate'] = new_df['is_duplicate']

In [14]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [15]:
from sklearn.model_selection import train_test_split
# Creating train-test split for model evaluation
X_train, X_test, y_train, y_test = train_test_split(
    temp_df.iloc[:, :-1].values, # Feature matrix 
    temp_df.iloc[:, -1].values, # Label vector
    test_size=0.2,  # 20% test data
    random_state=42) # Fixed seed for consistent results

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
# Initialize Random Forest classifier
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.746

In [17]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [19]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
# Training
xgb.fit(X_train,y_train)
y_pred = xgb.predict(X_test)
accuracy_score(y_test,y_pred)

0.7288333333333333

In [18]:
print(X_train.shape)
print(X_train.nbytes / 1e9, "GB") # agar 10GB+ hai toh 100% memory issue

(24000, 6000)
1.152 GB
